In [1]:
import pandas as pd
data = pd.read_csv('../data/train_master.csv')
data = data[['tweet_id', 'text', 'author_id', 'tw_date', 'year', 'AR', 'MB']]


In [2]:
data.dtypes

tweet_id      int64
text         object
author_id     int64
tw_date      object
year          int64
AR            int64
MB            int64
dtype: object

In [3]:
# Global var
test_size = 0.2
random_state = 2025

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(data.drop(columns=['AR', 'MB']),data[['AR', 'MB']], train_size=1-test_size, random_state=random_state, shuffle=True)
print(X_train.shape)

(2773, 5)


In [5]:
from scipy.stats import chi2_contingency

contingency_table = pd.crosstab(data['year'], data['AR'])

# Step 2: Perform the chi-squared test
chi2_stat, p_value, dof, expected = chi2_contingency(contingency_table)

# Step 3: Interpret the result
print(f"Chi-squared Statistic (rounded to 2 decimals): {round(chi2_stat, 2)}")
print(f"P-value (rounded to 6 decimals): {round(p_value,6)}")

if p_value < 0.05:
    print("Reject the null hypothesis: The variables are dependent (correlated).")
else:
    print("Fail to reject the null hypothesis: The variables are independent (not correlated).")

Chi-squared Statistic (rounded to 2 decimals): 271.75
P-value (rounded to 6 decimals): 0.0
Reject the null hypothesis: The variables are dependent (correlated).


# Logistic Regression

In [6]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Define categorical features
categorical_features = ['year', 'author_id']
lr_xtrain, lr_xtest, lr_ytrain, lr_ytest = X_train[categorical_features], X_test[categorical_features], y_train['AR'], y_test['AR']

# 1. Apply One-Hot Encoding manually
print("Applying One-Hot Encoding...")
one_hot_encoder = OneHotEncoder(handle_unknown='ignore')

# Fit and transform training data
lr_xtrain_encoded = one_hot_encoder.fit_transform(lr_xtrain)

# Transform test data using the fitted encoder
lr_xtest_encoded = one_hot_encoder.transform(lr_xtest)

print(f"Original feature shape: {lr_xtrain.shape}")
print(f"Encoded feature shape: {lr_xtrain_encoded.shape}")

Applying One-Hot Encoding...
Original feature shape: (2773, 2)
Encoded feature shape: (2773, 227)


In [ ]:
# 2. Train the Logistic Regression model directly
print("\nTraining the multi-class logistic regression model with multinomial strategy...")
model = LogisticRegression(
    penalty='l1',
    solver='saga',  # saga supports L1 regularization with multinomial
    class_weight='balanced',
    multi_class='multinomial',  # Uses multinomial loss
    C=0.1,
    random_state=2025,
    max_iter=1000  # saga might need more iterations
)

model.fit(lr_xtrain_encoded, lr_ytrain)
print("Training complete.")

# 3. Make predictions on the test data
lr_ypred = model.predict(lr_xtest_encoded)

# 4. Evaluate the model
print("\n--- Model Evaluation ---")
print("Classification Report:")
print(classification_report(lr_ytest, lr_ypred))

accuracy = model.score(lr_xtest_encoded, lr_ytest)
print(f"Accuracy: {accuracy:.2f}")

# 5. ACCESS MODEL COEFFICIENTS
print("\n--- Model Coefficients Analysis ---")

# Get feature names after one-hot encoding
feature_names = one_hot_encoder.get_feature_names_out(categorical_features)
print(f"Number of features after encoding: {len(feature_names)}")

# Get coefficients
coefficients = model.coef_
intercepts = model.intercept_

print(f"Coefficient matrix shape: {coefficients.shape}")
print(f"Number of classes: {len(model.classes_)}")
print(f"Classes: {model.classes_}")



Training the multi-class logistic regression model with multinomial strategy...
Training complete.

--- Model Evaluation ---
Classification Report:
              precision    recall  f1-score   support

           1       0.48      0.42      0.45       273
           2       0.38      0.27      0.31       148
           3       0.50      0.64      0.56       273

    accuracy                           0.48       694
   macro avg       0.45      0.44      0.44       694
weighted avg       0.47      0.48      0.47       694

Accuracy: 0.48

--- Model Coefficients Analysis ---
Number of features after encoding: 227
Coefficient matrix shape: (3, 227)
Number of classes: 3
Classes: [1 2 3]


In [8]:
# Display coefficients for each class
# Note: In multinomial logistic regression, coefficients are relative to the reference class
print("\n--- MULTINOMIAL COEFFICIENTS INTERPRETATION ---")
print("With multinomial strategy, coefficients represent the log-odds of each class")
print("relative to the other classes. The last class often serves as a reference.")

for i, class_label in enumerate(model.classes_):
    print(f"\n--- Coefficients for Class {class_label} ---")
    print(f"Intercept: {intercepts[i]:.4f}")
    
    # Create a DataFrame for better visualization
    coef_df = pd.DataFrame({
        'Feature': feature_names,
        'Coefficient': coefficients[i],
        'Abs_Coefficient': np.abs(coefficients[i])
    })
    
    # Sort by absolute coefficient value (descending)
    coef_df = coef_df.sort_values('Abs_Coefficient', ascending=False)
    
    # Display top 10 most important features (non-zero coefficients due to L1 regularization)
    non_zero_coefs = coef_df[coef_df['Coefficient'] != 0]
    print(f"Number of non-zero coefficients: {len(non_zero_coefs)}")
    
    if len(non_zero_coefs) > 0:
        print("Top 10 most important features:")
        print(non_zero_coefs.head(10).to_string(index=False, float_format='%.4f'))
    else:
        print("All coefficients are zero (very strong regularization)")
# 6. Coefficient differences between classes (multinomial-specific analysis)
print("\n--- MULTINOMIAL-SPECIFIC ANALYSIS ---")
print("Coefficient differences between classes (useful for understanding relative effects):")

classes = model.classes_
for i in range(len(classes)):
    for j in range(i+1, len(classes)):
        print(f"\n--- Class {classes[i]} vs Class {classes[j]} ---")
        
        # Calculate coefficient differences
        coef_diff = coefficients[i] - coefficients[j]
        
        # Create DataFrame for differences
        diff_df = pd.DataFrame({
            'Feature': feature_names,
            'Coefficient_Difference': coef_diff,
            'Abs_Difference': np.abs(coef_diff)
        }).sort_values('Abs_Difference', ascending=False)
        
        # Show top features that differentiate these two classes
        significant_diffs = diff_df[diff_df['Abs_Difference'] > 0.01]  # threshold for significance
        if len(significant_diffs) > 0:
            print(f"Top 5 features distinguishing Class {classes[i]} from Class {classes[j]}:")
            print(significant_diffs.head(5).to_string(index=False, float_format='%.4f'))
        else:
            print(f"No significant differences between Class {classes[i]} and Class {classes[j]}")

# 7. Overall feature importance across all classes
print("\n--- Overall Feature Importance (Max absolute coefficient across classes) ---")
max_abs_coefs = np.max(np.abs(coefficients), axis=0)
overall_importance = pd.DataFrame({
    'Feature': feature_names,
    'Max_Abs_Coefficient': max_abs_coefs
}).sort_values('Max_Abs_Coefficient', ascending=False)

# Show top 15 most important features overall
print("Top 15 most important features across all classes:")
print(overall_importance.head(15).to_string(index=False, float_format='%.4f'))

# 8. Summary statistics
print(f"\n--- Summary Statistics ---")
print(f"Total number of features: {len(feature_names)}")
print(f"Features with non-zero coefficients: {np.sum(max_abs_coefs > 0)}")
print(f"Sparsity ratio: {np.sum(max_abs_coefs == 0) / len(max_abs_coefs):.2f}")


--- MULTINOMIAL COEFFICIENTS INTERPRETATION ---
With multinomial strategy, coefficients represent the log-odds of each class
relative to the other classes. The last class often serves as a reference.

--- Coefficients for Class 1 ---
Intercept: 0.0226
Number of non-zero coefficients: 5
Top 10 most important features:
           Feature  Coefficient  Abs_Coefficient
author_id_29201047       0.4493           0.4493
         year_2019      -0.2292           0.2292
         year_2015      -0.0439           0.0439
author_id_29442313       0.0086           0.0086
         year_2021       0.0046           0.0046

--- Coefficients for Class 2 ---
Intercept: -0.1879
Number of non-zero coefficients: 8
Top 10 most important features:
                     Feature  Coefficient  Abs_Coefficient
                   year_2020       0.9427           0.9427
          author_id_18915145       0.2564           0.2564
                   year_2019       0.1557           0.1557
                   year_2016  

In [11]:
len(model.coef_[2])

227

In [12]:
X_train.author_id.value_counts().sort_values(ascending=False)

author_id
15745368              52
1074480192            52
13218102              49
18915145              48
17494010              47
                      ..
135297032              2
941080085121175552     2
409719505              2
15442036               1
18632809               1
Name: count, Length: 217, dtype: int64

# Word2Vec

In [10]:
! pip3 install -U gensim

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 24.0 MB 11.8 MB/s eta 0:00:01
     |████████████████████████████████| 61 kB 2.9 MB/s  eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from ftfy import fix_text
import re
# from preprocessing import preprocess_text
from scipy.sparse import hstack, csr_matrix
import warnings
warnings.filterwarnings('ignore')

# Define features
categorical_features = ['year', 'author_id']
text_column = 'text'

# Prepare data
wv_xtrain = X_train[categorical_features + [text_column]]
wv_xtest = X_test[categorical_features + [text_column]]
wv_ytrain = y_train['AR']
wv_ytest = y_test['AR']

print(f"Training data shape: {wv_xtrain.shape}")
print(f"Text column: {text_column}")

# ===== TEXT PREPROCESSING =====


print("\n===== PREPROCESSING TEXT =====")
# Preprocess training texts

def preprocess_text(text):
    """
    Clean tweet text by removing URLs, processing hashtags and mentions,
    fixing encoding issues, and normalizing whitespace.
    
    Args:
        text (str): Raw tweet text
        
    Returns:
        str: Cleaned tweet text
    """
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'#(\w+)', lambda m: ' '.join(re.findall(r'[A-Z][a-z]*|[a-z]+|\d+', m.group(1))),
                  text)  # Convert hashtags like #HelloWorld to Hello World

    # Error for #'s with shortcuts, i.e. #UTC is changed into `U T C`
    text = re.sub(r'@(\w+)', r'\1', text)  # Remove @ symbols from mentions
    text = fix_text(text)  # Fix encoding issues
    text = re.sub(r'[^\x00-\x7F]+', '', text)  # Remove non-ASCII characters
    # Remove extra spaces that may result from word removal
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'[\u200E\u200F\u202A-\u202E\u2066\u2069\u200B-\u200D\u00A0\u2000-\u200A\u00AD\uFFFD\u2063]', "", text)
    text = re.sub(r'\brt\b', "retweet", text, flags=re.IGNORECASE)  # Convert rt to retweet
    text = re.sub(r'\bw\b', "with", text, flags=re.IGNORECASE)  # Convert w to with

    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    text = simple_preprocess(text, deacc=True, min_len=2, max_len=15)
    return text

# Preprocess training texts
train_texts = wv_xtrain[text_column].apply(preprocess_text)
test_texts = wv_xtest[text_column].apply(preprocess_text)

# Remove empty documents
train_texts_clean = [text for text in train_texts if len(text) > 0]
print(f"Training documents after cleaning: {len(train_texts_clean)}")
print(f"Example preprocessed text: {train_texts_clean[0] if train_texts_clean else 'No valid texts'}")

# ===== TRAIN WORD2VEC MODEL =====
print("\n===== TRAINING WORD2VEC MODEL =====")
w2v_model = Word2Vec(
    sentences=train_texts_clean,
    vector_size=100,        # Dimension of word vectors
    window=5,               # Context window size
    min_count=2,            # Ignore words with frequency less than this
    workers=4,              # Number of CPU cores to use
    epochs=10,              # Number of training epochs
    sg=0,                   # 0 for CBOW, 1 for Skip-gram
    # random_state=42
)

print(f"Word2Vec model trained!")
print(f"Vocabulary size: {len(w2v_model.wv.key_to_index)}")
print(f"Vector dimension: {w2v_model.vector_size}")

# Show some example words
if len(w2v_model.wv.key_to_index) > 0:
    example_words = list(w2v_model.wv.key_to_index.keys())[:5]
    print(f"Example words in vocabulary: {example_words}")

# ===== CONVERT TEXTS TO VECTORS =====
def text_to_vector(text_tokens, model, vector_size):
    """
    Convert a list of tokens to a single vector by averaging word vectors
    """
    if len(text_tokens) == 0:
        return np.zeros(vector_size)
    
    # Get vectors for words that exist in vocabulary
    word_vectors = []
    for word in text_tokens:
        if word in model.wv:
            word_vectors.append(model.wv[word])
    
    if len(word_vectors) == 0:
        return np.zeros(vector_size)
    
    # Average the word vectors
    return np.mean(word_vectors, axis=0)

print("\n===== CONVERTING TEXTS TO VECTORS =====")
# Convert training texts to vectors
train_text_vectors = []
for text_tokens in train_texts:
    vector = text_to_vector(text_tokens, w2v_model, w2v_model.vector_size)
    train_text_vectors.append(vector)

train_text_vectors = np.array(train_text_vectors)

# Convert test texts to vectors
test_text_vectors = []
for text_tokens in test_texts:
    vector = text_to_vector(text_tokens, w2v_model, w2v_model.vector_size)
    test_text_vectors.append(vector)

test_text_vectors = np.array(test_text_vectors)

print(f"Training text vectors shape: {train_text_vectors.shape}")
print(f"Test text vectors shape: {test_text_vectors.shape}")

# ===== PROCESS CATEGORICAL FEATURES =====
print("\n===== PROCESSING CATEGORICAL FEATURES =====")
one_hot_encoder = OneHotEncoder(handle_unknown='ignore')

# Fit and transform training categorical data
train_categorical = wv_xtrain[categorical_features]
test_categorical = wv_xtest[categorical_features]

train_categorical_encoded = one_hot_encoder.fit_transform(train_categorical)
test_categorical = one_hot_encoder.transform(test_categorical)

print(f"Categorical features shape: {train_categorical.shape}")


Training data shape: (2773, 3)
Text column: text

===== PREPROCESSING TEXT =====
Training documents after cleaning: 2768
Example preprocessed text: ['another', 'great', 'with', 'works', 'rally', 'with', 'team', 'shelley', 'thank', 'you', 'hampshire', 'county', 'folks', 'for', 'coming', 'out', 'today', 'wvsen']

===== TRAINING WORD2VEC MODEL =====
Word2Vec model trained!
Vocabulary size: 4161
Vector dimension: 100
Example words in vocabulary: ['the', 'to', 'and', 'of', 'in']

===== CONVERTING TEXTS TO VECTORS =====
Training text vectors shape: (2773, 100)
Test text vectors shape: (694, 100)

===== PROCESSING CATEGORICAL FEATURES =====
Categorical features shape: (2773, 227)


In [20]:
X_train

,tweet_id,text,author_id,tw_date,year,clean_text
803,523220556859334656,Another great #WVWorks rally with #TeamShelley...,158890005,2014-10-17,2014,"[another, great, with, works, rally, with, tea..."
2919,1250082422877753345,Clean air and clean water are basic human nece...,521747968,2020-04-14,2020,"[clean, air, and, clean, water, are, basic, hu..."
946,661951497635278849,Honored to receive the Champion of Healthcare ...,78403308,2015-11-04,2015,"[honored, to, receive, the, champion, of, heal..."
3394,908759419055460353,Anthem staying in VA is great news. Proud of o...,172858784,2017-09-15,2017,"[anthem, staying, in, va, is, great, news, pro..."
437,1407386144468254723,No matter which way Democrats sell their elect...,1249982359,2021-06-22,2021,"[no, matter, which, way, democrats, sell, thei..."
...,...,...,...,...,...,...
1932,1080948298301796353,"Welcome to Congress, @JahanaHayesCT!\nhttps://...",150078976,2019-01-03,2019,"[welcome, to, congress, jahanahayesct]"
1235,1030625795306336256,.@Alaska_DHSS Dr. Butler at Wellness Summit 2....,2891210047,2018-08-18,2018,"[alaskadhss, dr, butler, at, wellness, summit,..."
323,1474136937434947594,The best protection against hospitalization an...,17494010,2021-12-23,2021,"[the, best, protection, against, hospitalizati..."
2910,1250076347030011905,We must grant small localities the same direct...,1099199839,2020-04-14,2020,"[we, must, grant, small, localities, the, same..."


In [27]:

# ===== COMBINE FEATURES =====
print("\n===== COMBINING FEATURES =====")
# Convert text vectors to sparse format for combining
train_text_sparse = csr_matrix(train_text_vectors)
test_text_sparse = csr_matrix(test_text_vectors)

# Combine categorical and text features
X_train_combined = hstack([train_categorical, train_text_sparse])
X_test_combined = hstack([test_categorical, test_text_sparse])

print(f"Combined training features shape: {X_train_combined.shape}")
print(f"Combined test features shape: {X_test_combined.shape}")

# ===== TRAIN LOGISTIC REGRESSION =====
print("\n===== TRAINING LOGISTIC REGRESSION =====")
model = LogisticRegression(
    penalty='l1',
    solver='saga',
    class_weight='balanced',
    multi_class='multinomial',
    C=0.1,
    random_state=42,
    max_iter=1000
)

model.fit(X_train_combined, wv_ytrain)
print("Training complete.")

# ===== EVALUATE MODEL =====
print("\n===== MODEL EVALUATION =====")
wv_ypred = model.predict(X_test_combined)

print("Classification Report:")
print(classification_report(wv_ytest, wv_ypred))

accuracy = model.score(X_test_combined, wv_ytest)
print(f"Accuracy: {accuracy:.4f}")

# ===== ANALYZE COEFFICIENTS =====
print("\n===== COEFFICIENT ANALYSIS =====")

# Create feature names
categorical_feature_names = one_hot_encoder.get_feature_names_out(categorical_features)
text_feature_names = [f'w2v_dim_{i}' for i in range(w2v_model.vector_size)]
all_feature_names = list(categorical_feature_names) + text_feature_names

coefficients = model.coef_
intercepts = model.intercept_

print(f"Total features: {len(all_feature_names)}")
print(f"Categorical features: {len(categorical_feature_names)}")
print(f"Word2Vec features: {len(text_feature_names)}")

# Analyze coefficients for each class
for i, class_label in enumerate(model.classes_):
    print(f"\n--- Class {class_label} Coefficients ---")
    print(f"Intercept: {intercepts[i]:.4f}")
    
    # Separate categorical and text coefficients
    cat_coefs = coefficients[i][:len(categorical_feature_names)]
    text_coefs = coefficients[i][len(categorical_feature_names):]
    
    # Top categorical features
    cat_df = pd.DataFrame({
        'Feature': categorical_feature_names,
        'Coefficient': cat_coefs,
        'Abs_Coefficient': np.abs(cat_coefs)
    }).sort_values('Abs_Coefficient', ascending=False)
    
    cat_non_zero = cat_df[cat_df['Coefficient'] != 0]
    if len(cat_non_zero) > 0:
        print(f"Top 5 categorical features:")
        print(cat_non_zero.head(5)[['Feature', 'Coefficient']].to_string(index=False, float_format='%.4f'))
    
    # Text feature analysis
    print(f"Text features - Mean abs coefficient: {np.mean(np.abs(text_coefs)):.4f}")
    print(f"Text features - Max abs coefficient: {np.max(np.abs(text_coefs)):.4f}")
    print(f"Text features - Non-zero coefficients: {np.sum(text_coefs != 0)}")

# ===== WORD2VEC MODEL INSIGHTS =====
print("\n===== WORD2VEC MODEL INSIGHTS =====")
if len(w2v_model.wv.key_to_index) > 5:
    # Find most similar words to some example words
    try:
        example_word = list(w2v_model.wv.key_to_index.keys())[0]
        similar_words = w2v_model.wv.most_similar(example_word, topn=5)
        print(f"Words most similar to '{example_word}':")
        for word, similarity in similar_words:
            print(f"  {word}: {similarity:.3f}")
    except:
        print("Not enough vocabulary for similarity analysis")

print(f"\nWord2Vec training complete! Text features integrated into logistic regression.")


===== COMBINING FEATURES =====
Combined training features shape: (2773, 327)
Combined test features shape: (694, 327)

===== TRAINING LOGISTIC REGRESSION =====
Training complete.

===== MODEL EVALUATION =====
Classification Report:
              precision    recall  f1-score   support

           1       0.48      0.42      0.45       273
           2       0.38      0.27      0.31       148
           3       0.50      0.64      0.56       273

    accuracy                           0.48       694
   macro avg       0.45      0.44      0.44       694
weighted avg       0.47      0.48      0.47       694

Accuracy: 0.4755

===== COEFFICIENT ANALYSIS =====
Total features: 327
Categorical features: 227
Word2Vec features: 100

--- Class 1 Coefficients ---
Intercept: 0.0216
Top 5 categorical features:
           Feature  Coefficient
author_id_29201047       0.4519
         year_2019      -0.2338
         year_2015      -0.0485
         year_2021       0.0121
author_id_29442313       0.010

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from ftfy import fix_text
import re
from scipy.sparse import hstack, csr_matrix
import warnings
warnings.filterwarnings('ignore')

# Define features
categorical_features = ['year', 'author_id']
text_column = 'text'

# Prepare data
wv_xtrain = X_train[categorical_features + [text_column]]
wv_xtest = X_test[categorical_features + [text_column]]
wv_ytrain = y_train['AR']
wv_ytest = y_test['AR']

print(f"Training data shape: {wv_xtrain.shape}")
print(f"Text column: {text_column}")

# ===== TEXT PREPROCESSING =====

def preprocess_text(text):
    """
    Clean tweet text by removing URLs, processing hashtags and mentions,
    fixing encoding issues, and normalizing whitespace.
    
    Args:
        text (str): Raw tweet text
        
    Returns:
        list: List of cleaned tokens
    """
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'#(\w+)', lambda m: ' '.join(re.findall(r'[A-Z][a-z]*|[a-z]+|\d+', m.group(1))),
                  text)  # Convert hashtags like #HelloWorld to Hello World

    # Error for #'s with shortcuts, i.e. #UTC is changed into `U T C`
    text = re.sub(r'@(\w+)', r'\1', text)  # Remove @ symbols from mentions
    text = fix_text(text)  # Fix encoding issues
    text = re.sub(r'[^\x00-\x7F]+', '', text)  # Remove non-ASCII characters
    # Remove extra spaces that may result from word removal
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'[\u200E\u200F\u202A-\u202E\u2066\u2069\u200B-\u200D\u00A0\u2000-\u200A\u00AD\uFFFD\u2063]', "", text)
    text = re.sub(r'\brt\b', "retweet", text, flags=re.IGNORECASE)  # Convert rt to retweet
    text = re.sub(r'\bw\b', "with", text, flags=re.IGNORECASE)  # Convert w to with

    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    text = simple_preprocess(text, deacc=True, min_len=2, max_len=15)
    return text

def preprocess_text_for_tfidf(text):
    """
    Preprocess text for TF-IDF (returns string instead of tokens)
    """
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'#(\w+)', lambda m: ' '.join(re.findall(r'[A-Z][a-z]*|[a-z]+|\d+', m.group(1))),
                  text)  # Convert hashtags like #HelloWorld to Hello World
    text = re.sub(r'@(\w+)', r'\1', text)  # Remove @ symbols from mentions
    text = fix_text(text)  # Fix encoding issues
    text = re.sub(r'[^\x00-\x7F]+', '', text)  # Remove non-ASCII characters
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'[\u200E\u200F\u202A-\u202E\u2066\u2069\u200B-\u200D\u00A0\u2000-\u200A\u00AD\uFFFD\u2063]', "", text)
    text = re.sub(r'\brt\b', "retweet", text, flags=re.IGNORECASE)
    text = re.sub(r'\bw\b', "with", text, flags=re.IGNORECASE)
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    
    # Return as string for TF-IDF
    tokens = simple_preprocess(text, deacc=True, min_len=2, max_len=15)
    return ' '.join(tokens)

print("\n===== PREPROCESSING TEXT =====")
# Preprocess training texts (tokens for Word2Vec)
train_texts = wv_xtrain[text_column].apply(preprocess_text)
test_texts = wv_xtest[text_column].apply(preprocess_text)

# Preprocess texts for TF-IDF (strings)
train_texts_tfidf = wv_xtrain[text_column].apply(preprocess_text_for_tfidf)
test_texts_tfidf = wv_xtest[text_column].apply(preprocess_text_for_tfidf)

# Remove empty documents
train_texts_clean = [text for text in train_texts if len(text) > 0]
print(f"Training documents after cleaning: {len(train_texts_clean)}")
print(f"Example preprocessed text: {train_texts_clean[0] if train_texts_clean else 'No valid texts'}")

# ===== TRAIN TF-IDF VECTORIZER =====
print("\n===== TRAINING TF-IDF VECTORIZER =====")
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,      # Limit vocabulary size
    min_df=2,               # Minimum document frequency
    max_df=0.8,             # Maximum document frequency
    ngram_range=(1, 2),     # Include unigrams and bigrams
    stop_words='english'    # Remove English stop words
)

# Fit TF-IDF on training texts
tfidf_matrix_train = tfidf_vectorizer.fit_transform(train_texts_tfidf)
tfidf_matrix_test = tfidf_vectorizer.transform(test_texts_tfidf)

print(f"TF-IDF vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")
print(f"TF-IDF matrix shape (train): {tfidf_matrix_train.shape}")

# Get TF-IDF feature names
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

# ===== TRAIN WORD2VEC MODEL =====
print("\n===== TRAINING WORD2VEC MODEL =====")
w2v_model = Word2Vec(
    sentences=train_texts_clean,
    vector_size=100,        # Dimension of word vectors
    window=5,               # Context window size
    min_count=2,            # Ignore words with frequency less than this
    workers=4,              # Number of CPU cores to use
    epochs=10,              # Number of training epochs
    sg=0,                   # 0 for CBOW, 1 for Skip-gram
)

print(f"Word2Vec model trained!")
print(f"Vocabulary size: {len(w2v_model.wv.key_to_index)}")
print(f"Vector dimension: {w2v_model.vector_size}")

# Show some example words
if len(w2v_model.wv.key_to_index) > 0:
    example_words = list(w2v_model.wv.key_to_index.keys())[:5]
    print(f"Example words in vocabulary: {example_words}")

# ===== TF-IDF WEIGHTED WORD2VEC AVERAGING =====
def text_to_tfidf_weighted_vector(text_tokens, doc_tfidf_vector, model, vectorizer, vector_size):
    """
    Convert a list of tokens to a single vector using TF-IDF weighted averaging
    
    Args:
        text_tokens: List of tokens from the document
        doc_tfidf_vector: Sparse TF-IDF vector for this document
        model: Word2Vec model
        vectorizer: TfidfVectorizer
        vector_size: Dimension of word vectors
    
    Returns:
        numpy array: TF-IDF weighted average of word vectors
    """
    if len(text_tokens) == 0:
        return np.zeros(vector_size)
    
    # Convert sparse vector to dense for easier access
    doc_tfidf_dense = doc_tfidf_vector.toarray().flatten()
    
    # Get word vectors and their TF-IDF weights
    weighted_vectors = []
    total_weight = 0.0
    
    for word in text_tokens:
        if word in model.wv:  # Word exists in Word2Vec model
            # Check if word exists in TF-IDF vocabulary
            if word in vectorizer.vocabulary_:
                tfidf_idx = vectorizer.vocabulary_[word]
                tfidf_weight = doc_tfidf_dense[tfidf_idx]
                
                if tfidf_weight > 0:  # Word has positive TF-IDF weight
                    word_vector = model.wv[word]
                    weighted_vectors.append(tfidf_weight * word_vector)
                    total_weight += tfidf_weight
    
    if len(weighted_vectors) == 0 or total_weight == 0:
        return np.zeros(vector_size)
    
    # Return weighted average
    return np.sum(weighted_vectors, axis=0) / total_weight

print("\n===== CONVERTING TEXTS TO TF-IDF WEIGHTED VECTORS =====")
# Convert training texts to TF-IDF weighted vectors
train_text_vectors = []
for i, text_tokens in enumerate(train_texts):
    doc_tfidf_vector = tfidf_matrix_train[i:i+1]  # Get TF-IDF vector for this document
    vector = text_to_tfidf_weighted_vector(text_tokens, doc_tfidf_vector, w2v_model, 
                                         tfidf_vectorizer, w2v_model.vector_size)
    train_text_vectors.append(vector)

train_text_vectors = np.array(train_text_vectors)

# Convert test texts to TF-IDF weighted vectors
test_text_vectors = []
for i, text_tokens in enumerate(test_texts):
    doc_tfidf_vector = tfidf_matrix_test[i:i+1]  # Get TF-IDF vector for this document
    vector = text_to_tfidf_weighted_vector(text_tokens, doc_tfidf_vector, w2v_model, 
                                         tfidf_vectorizer, w2v_model.vector_size)
    test_text_vectors.append(vector)

test_text_vectors = np.array(test_text_vectors)

print(f"Training text vectors shape: {train_text_vectors.shape}")
print(f"Test text vectors shape: {test_text_vectors.shape}")

# Check for any NaN or infinite values
print(f"Training vectors - NaN count: {np.sum(np.isnan(train_text_vectors))}")
print(f"Training vectors - Inf count: {np.sum(np.isinf(train_text_vectors))}")
print(f"Test vectors - NaN count: {np.sum(np.isnan(test_text_vectors))}")
print(f"Test vectors - Inf count: {np.sum(np.isinf(test_text_vectors))}")

# ===== PROCESS CATEGORICAL FEATURES =====
print("\n===== PROCESSING CATEGORICAL FEATURES =====")
one_hot_encoder = OneHotEncoder(handle_unknown='ignore')

# Fit and transform training categorical data
train_categorical = wv_xtrain[categorical_features]
test_categorical = wv_xtest[categorical_features]

train_categorical_encoded = one_hot_encoder.fit_transform(train_categorical)
test_categorical_encoded = one_hot_encoder.transform(test_categorical)

print(f"Categorical features shape: {train_categorical_encoded.shape}")

# ===== COMBINE FEATURES =====
print("\n===== COMBINING FEATURES =====")
# Convert text vectors to sparse format for combining
train_text_sparse = csr_matrix(train_text_vectors)
test_text_sparse = csr_matrix(test_text_vectors)

# Combine categorical and text features
X_train_combined = hstack([train_categorical_encoded, train_text_sparse])
X_test_combined = hstack([test_categorical_encoded, test_text_sparse])

print(f"Combined training features shape: {X_train_combined.shape}")
print(f"Combined test features shape: {X_test_combined.shape}")

# ===== TRAIN LOGISTIC REGRESSION =====
print("\n===== TRAINING LOGISTIC REGRESSION =====")
model = LogisticRegression(
    penalty='l1',
    solver='saga',
    class_weight='balanced',
    multi_class='multinomial',
    C=0.1,
    random_state=42,
    max_iter=1000
)

model.fit(X_train_combined, wv_ytrain)
print("Training complete.")

# ===== EVALUATE MODEL =====
print("\n===== MODEL EVALUATION =====")
wv_ypred = model.predict(X_test_combined)

print("Classification Report:")
print(classification_report(wv_ytest, wv_ypred))

accuracy = model.score(X_test_combined, wv_ytest)
print(f"Accuracy: {accuracy:.4f}")

# ===== ANALYZE COEFFICIENTS =====
print("\n===== COEFFICIENT ANALYSIS =====")

# Create feature names
categorical_feature_names = one_hot_encoder.get_feature_names_out(categorical_features)
text_feature_names = [f'w2v_tfidf_dim_{i}' for i in range(w2v_model.vector_size)]
all_feature_names = list(categorical_feature_names) + text_feature_names

coefficients = model.coef_
intercepts = model.intercept_

print(f"Total features: {len(all_feature_names)}")
print(f"Categorical features: {len(categorical_feature_names)}")
print(f"TF-IDF weighted Word2Vec features: {len(text_feature_names)}")

# Analyze coefficients for each class
for i, class_label in enumerate(model.classes_):
    print(f"\n--- Class {class_label} Coefficients ---")
    print(f"Intercept: {intercepts[i]:.4f}")
    
    # Separate categorical and text coefficients
    cat_coefs = coefficients[i][:len(categorical_feature_names)]
    text_coefs = coefficients[i][len(categorical_feature_names):]
    
    # Top categorical features
    cat_df = pd.DataFrame({
        'Feature': categorical_feature_names,
        'Coefficient': cat_coefs,
        'Abs_Coefficient': np.abs(cat_coefs)
    }).sort_values('Abs_Coefficient', ascending=False)
    
    cat_non_zero = cat_df[cat_df['Coefficient'] != 0]
    if len(cat_non_zero) > 0:
        print(f"Top 5 categorical features:")
        print(cat_non_zero.head(5)[['Feature', 'Coefficient']].to_string(index=False, float_format='%.4f'))
    
    # Text feature analysis
    print(f"TF-IDF weighted text features:")
    print(f"  Mean abs coefficient: {np.mean(np.abs(text_coefs)):.4f}")
    print(f"  Max abs coefficient: {np.max(np.abs(text_coefs)):.4f}")
    print(f"  Non-zero coefficients: {np.sum(text_coefs != 0)}")

# ===== COMPARISON WITH SIMPLE AVERAGING =====
print("\n===== COMPARISON: TF-IDF WEIGHTED vs SIMPLE AVERAGING =====")
print("TF-IDF weighting gives more importance to:")
print("1. Words that are frequent in the document (high TF)")
print("2. Words that are rare across the corpus (high IDF)")
print("3. This should better capture document-specific important words")
print("4. Compared to simple averaging which treats all words equally")

print(f"\nWord2Vec + TF-IDF weighted averaging complete!")

Training data shape: (2773, 3)
Text column: text

===== PREPROCESSING TEXT =====
Training documents after cleaning: 2768
Example preprocessed text: ['another', 'great', 'with', 'works', 'rally', 'with', 'team', 'shelley', 'thank', 'you', 'hampshire', 'county', 'folks', 'for', 'coming', 'out', 'today', 'wvsen']

===== TRAINING TF-IDF VECTORIZER =====
TF-IDF vocabulary size: 5000
TF-IDF matrix shape (train): (2773, 5000)

===== TRAINING WORD2VEC MODEL =====
Word2Vec model trained!
Vocabulary size: 4161
Vector dimension: 100
Example words in vocabulary: ['the', 'to', 'and', 'of', 'in']

===== CONVERTING TEXTS TO TF-IDF WEIGHTED VECTORS =====
Training text vectors shape: (2773, 100)
Test text vectors shape: (694, 100)
Training vectors - NaN count: 0
Training vectors - Inf count: 0
Test vectors - NaN count: 0
Test vectors - Inf count: 0

===== PROCESSING CATEGORICAL FEATURES =====
Categorical features shape: (2773, 227)

===== COMBINING FEATURES =====
Combined training features shape: (2773

# Trees

In [43]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
import numpy as np

categorical_features = ['year', 'author_id']
xgb_xtrain, xgb_xtest = X_train[categorical_features], X_test[categorical_features]


for col in categorical_features:
    xgb_xtrain[col] = xgb_xtrain[col].astype('category')
    xgb_xtest[col] = xgb_xtest[col].astype('category')



test_params = {
    "max_depth": np.arange(3, 12, 3),
    "eta": np.array([0.001, 0.01, 0.05, 0.1, 0.3, 0.5]),
}


xgb_cv = GridSearchCV(
    estimator=xgb.XGBClassifier(
        n_estimators=100,
        objective="multi:softprob",  # or "multi:softmax" 
        eval_metric="mlogloss",      # or "merror"
        subsample=0.5,
        min_child_weight=50,
        random_state=random_state,
        enable_categorical=True
    ),
    param_grid=test_params,
    scoring="f1_macro",  # or "f1_micro", "accuracy"
    cv=5  # specify cross-validation folds
)

xgb_ytrain, xgb_ytest = lr_ytrain-1, lr_ytest - 1

xgb_cv.fit(xgb_xtrain, xgb_ytrain)

/var/folders/_5/c3vpzww13z34kn9h6pvd032h0000gn/T/ipykernel_93875/1972527133.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  xgb_xtrain[col] = xgb_xtrain[col].astype('category')
/var/folders/_5/c3vpzww13z34kn9h6pvd032h0000gn/T/ipykernel_93875/1972527133.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  xgb_xtest[col] = xgb_xtest[col].astype('category')


GridSearchCV(cv=5,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=True,
                                     eval_metric='mlogloss', feature_types=None,
                                     gamma=None, grow_policy=None,
                                     importance_type=None,
                                     interaction_constraints=None,
                                     learning_rate=...
                                     max_cat_threshold=None,
                                     max_cat_to_onehot=None,
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=Non

In [44]:
cv_df = pd.DataFrame(xgb_cv.cv_results_)[
    ["params", "mean_test_score", "rank_test_score"]
]
cv_df = pd.concat(
    [cv_df["params"].apply(pd.Series), cv_df.drop(["params"], axis=1)], axis=1
)
cv_df.sort_values("rank_test_score")

,eta,max_depth,mean_test_score,rank_test_score
17,0.500,9.0,0.435141,1
0,0.001,3.0,0.432043,2
6,0.050,3.0,0.431022,3
2,0.001,9.0,0.430762,4
3,0.010,3.0,0.430762,4
1,0.001,6.0,0.430762,4
4,0.010,6.0,0.430731,7
5,0.010,9.0,0.430731,7
8,0.050,9.0,0.428969,9
7,0.050,6.0,0.428835,10


In [54]:
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# 1. Train the model
model = xgb.XGBClassifier(
    n_estimator=100,
    learning_rate=0.5,  # 'eta' is deprecated, use 'learning_rate'
    max_depth=9,
    objective="multi:softprob",
    eval_metric="mlogloss",
    subsample=0.5,
    min_child_weight=50,
    random_state=random_state,
    enable_categorical=True
)

model.fit(xgb_xtrain, xgb_ytrain)
print("Training complete.")

# 2. Make predictions on the test data
xgb_ypred = model.predict(xgb_xtest)

# 3. Evaluate the model
print("\n--- Model Evaluation ---")
print("Classification Report:")
print(classification_report(xgb_ytest, xgb_ypred))

# Use accuracy_score instead of model.score for clarity
accuracy = accuracy_score(xgb_ytest, xgb_ypred)
print(f"Accuracy: {accuracy:.2f}")

# 4. FEATURE IMPORTANCE (XGBoost doesn't have coefficients like linear models)
print("\n--- Feature Importance Analysis ---")

# XGBoost uses feature_importances_, not coef_
feature_importances = model.feature_importances_
feature_names = xgb_xtrain.columns.tolist()

print(f"Number of features: {len(feature_names)}")
print(f"Feature names: {feature_names}")

# Display feature importances
importance_dict = dict(zip(feature_names, feature_importances))
print("\nFeature Importances:")
for feature, importance in sorted(importance_dict.items(), key=lambda x: x[1], reverse=True):
    print(f"{feature}: {importance:.4f}")

print(f"Number of classes: {len(model.classes_)}")
print(f"Classes: {model.classes_}")

# 6. Prediction probabilities (useful for multi-class)
xgb_ypred_proba = model.predict_proba(xgb_xtest)
print(f"\nPrediction probabilities shape: {xgb_ypred_proba.shape}")


/Users/alexandermichaeltjhin/.local/lib/python3.9/site-packages/xgboost/core.py:160: UserWarning: [21:29:25] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:742: 
Parameters: { "n_estimator" } are not used.

  warnings.warn(smsg, UserWarning)


Training complete.

--- Model Evaluation ---
Classification Report:
              precision    recall  f1-score   support

           0       0.46      0.42      0.44       273
           1       0.38      0.22      0.28       148
           2       0.48      0.62      0.54       273

    accuracy                           0.46       694
   macro avg       0.44      0.42      0.42       694
weighted avg       0.45      0.46      0.45       694

Accuracy: 0.46

--- Feature Importance Analysis ---
Number of features: 2
Feature names: ['year', 'author_id']

Feature Importances:
author_id: 0.5302
year: 0.4698
Number of classes: 3
Classes: [0 1 2]

Prediction probabilities shape: (694, 3)
